# Local PySpark Replacement for Google Colab

This notebook replaces `UsePySparkatGoogleColab.txt` with a fully local workflow on Windows. No Colab runtime, no `files.upload()` dialog, no re-uploading data every session.

**Prerequisites (one-time setup):**
1. Install Adoptium Temurin JDK 17 (or 11): https://adoptium.net/
2. Add the JDK `bin` folder to `PATH`, or let the cell below set `JAVA_HOME`.
3. `pip install pyspark==3.5.0 pandas` in your Python environment.

The Colab instructions install Java + PySpark every session and use `files.upload()`. Locally we skip both: Java is already on disk, and data files live alongside the notebook.

## 1. Environment setup

Point `JAVA_HOME` at a real JDK. Edit the candidate list if yours lives elsewhere. Spark 3.5 works with JDK 8, 11, or 17.

In [ ]:
import os, sys

JAVA_HOME_CANDIDATES = [
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
    r"C:\Program Files\Eclipse Adoptium\jdk-25.0.2.10-hotspot",
]
for candidate in JAVA_HOME_CANDIDATES:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        os.environ["PATH"] = candidate + r"\bin;" + os.environ["PATH"]
        break

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("JAVA_HOME =", os.environ.get("JAVA_HOME", "<not set>"))

## 2. Create a Spark session

Local mode, all cores. No cluster, no driver memory negotiation.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("StevensGraduateCourse")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready.")

## 3. Load data from disk

Colab uses `files.upload()`. Locally we read straight from the `supplemental_materials/Data/` folder that lives next to this notebook. `re_u.data` is tab-separated with four columns: userID, itemID, rating, timestamp.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

DATA_DIR = os.path.join(os.getcwd(), "supplemental_materials", "Data")
re_u_path = os.path.join(DATA_DIR, "re_u.data")
print("Reading", re_u_path)

schema = StructType([
    StructField("userID", IntegerType(), True),
    StructField("itemID", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", IntegerType(), True),
])

ratings = spark.read.csv(re_u_path, sep="\t", schema=schema)
ratings.show(5)
print("rows:", ratings.count())

## 4. Minimal ALS smoke test

Confirms the full stack works end-to-end: DataFrame to ALS fit to predict to RMSE. If this cell prints an RMSE, PySpark is fully operational and HW8 Part 2 sweeps can reuse the same session.

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

train, test = ratings.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    maxIter=5,
    rank=5,
    regParam=0.1,
    userCol="userID",
    itemCol="itemID",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
)
model = als.fit(train)
preds = model.transform(test)

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(preds)
mse = rmse ** 2
print(f"RMSE = {rmse:.4f}   MSE = {mse:.4f}")
preds.show(5)

## 5. Teardown

Only needed if you plan to create a second session in the same kernel.

In [ ]:
# spark.stop()